# Lab 7.2: LIME for a Retail Churn Model, and How Far to Trust It

Run the cells from top to bottom. You do not need to change any code until the optional last section. Each checkpoint tells you what you should see.

Data: Online Retail II (Chen, 2012), UCI Machine Learning Repository, CC BY 4.0, https://doi.org/10.24432/C5CG6D. About a million invoice lines from a UK online gift retailer, December 2009 to December 2011. We turn it into one row per customer and ask: will this customer place no order in the next six months?

This notebook reproduces the data case study of Chapter 7.

## 1. Build the customer table

The first run downloads about 45 MB and reads an Excel file, which takes one to two minutes. Later runs use a cached CSV.

In [1]:
import os, zipfile, urllib.request, tempfile, warnings, itertools
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

UCI = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
CUTOFF, HORIZON = pd.Timestamp("2011-06-09"), 183

def load_transactions():
    """Online Retail II (Chen, 2012; UCI, CC BY 4.0). Downloads about 45 MB once to a temporary
    folder and caches a compressed CSV there. Offline: put online_retail_II.xlsx next to this notebook."""
    cache = Path(tempfile.gettempdir()) / "xai-ch07"; cache.mkdir(exist_ok=True)
    csv = cache / "online_retail_II.csv.gz"
    if not csv.exists():
        xlsx = Path("online_retail_II.xlsx")
        if not xlsx.exists():
            xlsx = cache / "online_retail_II.xlsx"
            if not xlsx.exists():
                z = cache / "oretail.zip"; urllib.request.urlretrieve(UCI, z)
                zipfile.ZipFile(z).extractall(cache)
        sheets = pd.read_excel(xlsx, sheet_name=None, dtype={"Invoice": str, "StockCode": str})
        pd.concat(sheets.values(), ignore_index=True).to_csv(csv, index=False)
    d = pd.read_csv(csv, dtype={"Invoice": str, "StockCode": str}, parse_dates=["InvoiceDate"])
    d = d.dropna(subset=["Customer ID"]).copy()
    d["cid"] = d["Customer ID"].astype(int)
    d["value"] = d["Quantity"] * d["Price"]
    d["is_return"] = d["Invoice"].str.startswith("C")
    return d

def churn_table(d):
    """One row per customer who bought in the 12 months before the cutoff.
    churn = 1 if the customer places no order in the 183 days after the cutoff."""
    past = d[d.InvoiceDate < CUTOFF]
    future = d[(d.InvoiceDate >= CUTOFF) & (d.InvoiceDate < CUTOFF + pd.Timedelta(days=HORIZON)) & ~d.is_return]
    buys, rets = past[~past.is_return], past[past.is_return]
    year = buys[buys.InvoiceDate >= CUTOFF - pd.Timedelta(days=365)]
    active = np.sort(year.cid.unique())
    g, g12 = buys[buys.cid.isin(active)].groupby("cid"), year.groupby("cid")
    f = pd.DataFrame(index=pd.Index(active, name="cid"))
    f["recency_days"] = (CUTOFF - g.InvoiceDate.max()).dt.days
    f["tenure_days"] = (CUTOFF - g.InvoiceDate.min()).dt.days
    f["orders_12m"] = g12.Invoice.nunique()
    f["spend_12m"] = g12.value.sum().clip(lower=0)
    f["avg_order_value"] = f.spend_12m / f.orders_12m
    f["products_12m"] = g12.StockCode.nunique()
    q = year[year.InvoiceDate >= CUTOFF - pd.Timedelta(days=90)]
    f["orders_90d"] = q.groupby("cid").Invoice.nunique().reindex(f.index).fillna(0)
    rv = rets[rets.InvoiceDate >= CUTOFF - pd.Timedelta(days=365)].groupby("cid").value.sum().abs().reindex(f.index).fillna(0)
    f["return_share"] = (rv / (f.spend_12m + rv)).fillna(0)
    country = buys.groupby("cid").Country.agg(lambda s: s.mode().iloc[0])
    f["uk"] = (country.reindex(f.index) == "United Kingdom").astype(int)
    f["churn"] = (~f.index.isin(future.cid.unique())).astype(int)
    return f

d = load_transactions()
f = churn_table(d)
print("transactions with a customer ID:", len(d))
print("customers:", len(f), " churn rate: %.3f" % f.churn.mean())
f.head()

transactions with a customer ID: 824364
customers: 4335  churn rate: 0.427


,recency_days,tenure_days,orders_12m,spend_12m,avg_order_value,products_12m,orders_90d,return_share,uk,churn
cid,,,,,,,,,,
12346,141,541,2,77325.91,38662.9550,20,0.0,0.500619,1,1
12347,62,220,4,3146.75,786.6875,90,1.0,0.000000,0,0
12348,64,254,4,1709.40,427.3500,25,1.0,0.000000,0,0
12349,223,405,1,1402.62,1402.6200,55,0.0,0.000000,0,0
12350,126,126,1,334.40,334.4000,17,0.0,0.000000,0,1


**Checkpoint 1.** 4,335 customers and a churn rate of about 0.427. Look at the definitions in `churn_table`: which features are bound to each other by definition? Days since the last order can never exceed days since the first order, and orders in 90 days can never exceed orders in 12 months. For a customer with one order in 12 months, spend equals average order value, and for about 70 percent of them recency equals tenure (the others bought before the 12-month window).

In [2]:
one = f[f.orders_12m == 1]
print("one-order customers:", len(one))
print("recency equals tenure for one-order customers:", (one.recency_days == one.tenure_days).mean().round(3))
print("recency never exceeds tenure in real data:", (f.recency_days <= f.tenure_days).all())
print("orders_90d never exceeds orders_12m:", (f.orders_90d <= f.orders_12m).all())

one-order customers: 1552
recency equals tenure for one-order customers: 0.701
recency never exceeds tenure in real data: True
orders_90d never exceeds orders_12m: True


## 2. Train the black box

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X, y = f.drop(columns="churn"), f.churn
names = list(X.columns)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, random_state=0).fit(Xtr, ytr)
logit = LogisticRegression(max_iter=2000).fit(np.log1p(Xtr.clip(lower=0)), ytr)
print("test AUC, boosted trees:       %.3f" % roc_auc_score(yte, model.predict_proba(Xte)[:, 1]))
print("test AUC, logistic on log(1+x): %.3f" % roc_auc_score(yte, logit.predict_proba(np.log1p(Xte.clip(lower=0)))[:, 1]))
predict_proba = lambda a: model.predict_proba(pd.DataFrame(a, columns=names))

test AUC, boosted trees:       0.784
test AUC, logistic on log(1+x): 0.787


**Checkpoint 2.** Boosted trees reach a test AUC of about 0.784; a logistic regression on log-transformed features reaches about 0.787. Keep that in mind: Chapter 5 asks whether you needed the black box at all. We explain the boosted model anyway, because that is the situation LIME was built for.

## 3. LIME by hand

Before using a library, build the method yourself on one customer: sample, weight, fit. This version perturbs the standardized features with Gaussian noise around the customer, as in the continuous variant of tabular LIME.

In [4]:
def lime_by_hand(x, n=5000, width=None, seed=0):
    rng = np.random.default_rng(seed)
    mu, sd = Xtr.mean().values, Xtr.std().values
    xs = (x - mu) / sd                                   # standardize the customer
    Zs = xs + rng.normal(0, 1, (n, len(x)))              # perturb in standardized space
    Z = Zs * sd + mu                                     # back to original units for the model
    width = width or 0.75 * np.sqrt(len(x))              # the common default kernel width
    w = np.sqrt(np.exp(-((Zs - xs) ** 2).sum(1) / width ** 2))
    A = np.column_stack([np.ones(n), Zs - xs])
    sw = np.sqrt(w)[:, None]
    beta = np.linalg.lstsq(A * sw, predict_proba(Z)[:, 1] * sw[:, 0], rcond=None)[0]
    return pd.Series(beta[1:], index=names), beta[0]

x = X.loc[16027].values
print("model churn probability for customer 16027: %.3f" % predict_proba(x[None])[0, 1])
coef, intercept = lime_by_hand(x)
print("surrogate at the customer: %.3f" % intercept)
coef.sort_values(key=abs, ascending=False).round(3)

model churn probability for customer 16027: 0.511


surrogate at the customer: 0.295


orders_12m        -0.149
products_12m      -0.075
spend_12m         -0.068
recency_days       0.058
tenure_days       -0.047
orders_90d        -0.039
return_share       0.020
avg_order_value    0.019
uk                -0.004
dtype: float64

**Checkpoint 3.** The weights are slopes per standard deviation of each feature, around this customer. Write down the three largest. Notice that the perturbations here include impossible customers (negative orders, recency above tenure); section 6 counts them for the library version.

## 4. LIME with the `lime` library (tabular, discretized)

In [5]:
from lime.lime_tabular import LimeTabularExplainer

def lime_explainer(seed=0, kernel_width=None):
    return LimeTabularExplainer(Xtr.values, feature_names=names, class_names=["stay", "churn"],
                                discretize_continuous=True, random_state=seed, kernel_width=kernel_width)

exp = lime_explainer().explain_instance(x, predict_proba, num_features=len(names), num_samples=5000)
print("model: %.3f   surrogate at the instance: %.3f   weighted R2: %.3f" % (predict_proba(x[None])[0, 1], exp.local_pred[0], exp.score))
for cond, wt in exp.as_list():
    print(f"{wt:+.3f}  {cond}")

model: 0.511   surrogate at the instance: 0.344   weighted R2: 0.125
-0.118  orders_12m > 4.75
-0.078  31.00 < recency_days <= 106.00
+0.066  products_12m <= 16.00
+0.052  306.84 < spend_12m <= 671.99
-0.045  tenure_days > 495.00
+0.040  0.00 < orders_90d <= 1.00
+0.024  return_share > 0.01
-0.004  avg_order_value <= 184.33
+0.000  uk <= 1.00


**Checkpoint 4.** The model gives 0.511; the surrogate's own prediction at the instance is about 0.34 and its weighted R squared about 0.12. The two largest weights (`orders_12m > 4.75`, `31 < recency_days <= 106`) push *toward staying*, while the model flags the customer as a churn risk. Ask yourself: if an agent read the top two "reasons" to this customer, would they explain the flag?

## 5. Stability: rerun with different seeds

In [6]:
def top_k(cid, seed, n, k=3, kernel_width=None):
    e = lime_explainer(seed, kernel_width).explain_instance(X.loc[cid].values, predict_proba, num_features=k, num_samples=n)
    return [names[j] for j, _ in e.as_map()[1]]

def jaccard(a, b):
    return len(set(a) & set(b)) / len(set(a) | set(b))

rows = []
for cid in [16027, 14933, 17726, 14203]:
    for n in [250, 1000, 5000]:
        tops = [top_k(cid, s, n) for s in range(30)]
        J = np.mean([jaccard(a, b) for a, b in itertools.combinations(tops, 2)])
        rows.append((cid, n, round(J, 2)))
pd.DataFrame(rows, columns=["customer", "samples", "mean top-3 Jaccard"]).pivot(index="customer", columns="samples")

mean top-3 Jaccard            
samples                250   1000  5000
customer                               
14203                  0.62  0.63  0.68
14933                  0.63  0.72  0.88
16027                  0.47  0.72  0.91
17726                  0.56  0.77  0.91

**Checkpoint 5.** Agreement rises with the number of samples for most customers (about 0.47 to 0.91 for customer 16027 between 250 and 5,000 samples), but customer 14203, a one-order buyer, stays near 0.68 even at 5,000. More samples do not fix everything: when features are redundant, credit moves between them.

## 6. Where do LIME's samples land?

In [7]:
ex = lime_explainer()
_, samples = ex._LimeTabularExplainer__data_inverse(X.loc[14203].values, 5000)   # the library's own sampler
S = pd.DataFrame(samples, columns=names)
print("samples with last order before first order: %.3f" % (S.recency_days > S.tenure_days + 0.5).mean())
print("samples with more orders in 90 days than in 12 months: %.3f" % (S.orders_90d > S.orders_12m + 0.5).mean())
print("either: %.3f" % ((S.recency_days > S.tenure_days + 0.5) | (S.orders_90d > S.orders_12m + 0.5)).mean())

samples with last order before first order: 0.148
samples with more orders in 90 days than in 12 months: 0.219
either: 0.336


**Checkpoint 6.** About 15 percent of the samples have a last order before the first order, about 22 percent have more orders in 90 days than in 12 months, and about a third break at least one of the two rules. None of these customers can exist. The surrogate is fitted partly to the model's answers on impossible inputs (Chapter 3 called them off-manifold).

## 7. Kernel width

In [8]:
for kw in [0.5, 1, 0.75 * np.sqrt(len(names)), 5, 20]:
    e = lime_explainer(0, kw).explain_instance(X.loc[16027].values, predict_proba, num_features=3, num_samples=5000)
    print(f"kernel width {kw:5.2f}:", [(c, round(w, 3)) for c, w in e.as_list()], " R2 %.2f" % e.score)

kernel width  0.50: [('orders_12m > 4.75', -0.045), ('products_12m <= 16.00', 0.033), ('31.00 < recency_days <= 106.00', -0.031)]  R2 0.06


kernel width  1.00: [('orders_12m > 4.75', -0.136), ('31.00 < recency_days <= 106.00', -0.089), ('products_12m <= 16.00', 0.068)]  R2 0.15


kernel width  2.25: [('orders_12m > 4.75', -0.117), ('31.00 < recency_days <= 106.00', -0.077), ('products_12m <= 16.00', 0.067)]  R2 0.10


kernel width  5.00: [('orders_12m > 4.75', -0.114), ('31.00 < recency_days <= 106.00', -0.075), ('products_12m <= 16.00', 0.067)]  R2 0.09


kernel width 20.00: [('orders_12m > 4.75', -0.114), ('31.00 < recency_days <= 106.00', -0.075), ('products_12m <= 16.00', 0.066)]  R2 0.09


**Checkpoint 7.** From width 1 upward the top three and their weights barely move. At width 0.5 the same three features appear, but every weight shrinks to about a third (for example -0.045 instead of -0.117 for `orders_12m`) and the weighted R squared falls to 0.06: almost all samples get a tiny weight, and the ridge penalty in the surrogate dominates the fit (Tan et al., 2023, describe this mechanism). The width is a choice about what "local" means; record it with every explanation.

## 8. Optional: your turn

Pick a customer of your own from the test set with a churn probability between 0.4 and 0.6 (near the decision threshold). Run sections 4 to 7 for that customer. Is the explanation stable enough to be read to the customer by an agent?

In [9]:
p = model.predict_proba(Xte)[:, 1]
candidates = Xte[(p > 0.4) & (p < 0.6)].index[:10]
list(candidates)

[15922, 17376, 17905, 17101, 16720, 17978, 16138, 12842, 16552, 12529]